In [ ]:
import os
import gymnasium as gym
import numpy as np
import json
from collections import deque

# ĐỔI THƯ VIỆN SANG SB3-CONTRIB
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList, BaseCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv

from mahjong_env import MahjongEnv

class SlidingWindowStatsCallback(BaseCallback):
    def __init__(self, window_size=10, verbose=1):
        super(SlidingWindowStatsCallback, self).__init__(verbose)
        self.window_size = window_size
        self.history_window = deque(maxlen=window_size)
        self.current_iteration_wins = []
        self.env_step_counts = None
        self.iteration_count = 0 # Đếm số vòng học

    def _on_training_start(self) -> None:
        n_envs = self.training_env.num_envs
        self.env_step_counts = np.zeros(n_envs, dtype=int)

    def _on_step(self) -> bool:
        self.env_step_counts += 1
        for i, done in enumerate(self.locals['dones']):
            if done:
                info = self.locals['infos'][i]
                if info.get('shanten') == -1: 
                    self.current_iteration_wins.append(self.env_step_counts[i])
                self.env_step_counts[i] = 0
        return True

    def _on_rollout_end(self) -> None:
        self.iteration_count += 1
        self.history_window.append(self.current_iteration_wins)
        
        all_wins_in_window = []
        for iteration_wins in self.history_window:
            all_wins_in_window.extend(iteration_wins)
            
        # Nếu verbose = 1, in ra terminal một cái bảng nhỏ gọn
        if self.verbose > 0:
            print("\n" + "="*50)
            print(f"🔄 HOÀN TẤT ITERATION THỨ {self.iteration_count}")
            print(f"├─ Trong 2048 bước vừa qua: AI tới bài {len(self.current_iteration_wins)} lần.")
            
            if len(all_wins_in_window) > 0:
                avg_steps = np.mean(all_wins_in_window)
                print(f"└─ Thống kê {len(self.history_window)} vòng gần nhất: Tổng {len(all_wins_in_window)} ván thắng | Tốc độ: {avg_steps:.1f} bước/ván.")
            else:
                print(f"└─ Thống kê {len(self.history_window)} vòng gần nhất: AI đang tìm cách hạ Shanten...")
            print("="*50 + "\n")
                
        self.current_iteration_wins = []

# Tạo một hàm bọc (factory function) để sinh ra môi trường đã được mask
def make_masked_env():
    def _init():
        env = MahjongEnv()
        env = Monitor(env)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

# Hàm lấy mask từ môi trường
def mask_fn(env: gym.Env) -> np.ndarray:
    # Lấy action_mask từ class MahjongEnv của bạn
    return env.unwrapped._action_mask()

import os
import json
import numpy as np
from stable_baselines3.common.callbacks import CheckpointCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker

# (Đảm bảo bạn vẫn giữ các hàm mask_fn, make_masked_env, và class Callback ở trên)

def main():
    # ==========================================
    # 1. KHỞI TẠO MÔI TRƯỜNG ĐA LUỒNG (TRAINING)
    # ==========================================
    n_envs = 8
    print(f"Khởi tạo {n_envs} môi trường chạy song song...\n")
    
    # Mở 8 tiến trình để gom dữ liệu cực nhanh
    env = SubprocVecEnv([make_masked_env() for _ in range(n_envs)])

    # ==========================================
    # 2. KHỞI TẠO HOẶC TẢI MODEL PPO ĐỂ HỌC TIẾP
    # ==========================================
    model_path = "./models_1/checkpoints_1/mahjong_model_1000000_steps.zip"
    
    if os.path.exists(model_path):
        print(f"\n[INFO] Đã tìm thấy model cũ tại '{model_path}'. Tiến hành tải và học tiếp...")
        # Tải model cũ lên và gán môi trường mới vào
        model = MaskablePPO.load(model_path, env=env)
        
        # (Tùy chọn) Giảm learning_rate khi học tiếp để AI không bị "tẩy não" quên kiến thức cũ
        # model.learning_rate = 1e-4 
    else:
        print("\n[INFO] Không tìm thấy model cũ. Khởi tạo AI học từ đầu...")
        model = MaskablePPO(
            "MlpPolicy", 
            env, 
            device="cpu", 
            learning_rate=3e-4,
            n_steps=256, 
            batch_size=128, 
            verbose=1,
            tensorboard_log="./logs_tensorboard/"
        )

    # ==========================================
    # 3. THIẾT LẬP CALLBACKS
    # ==========================================
    os.makedirs("./models/checkpoints/", exist_ok=True)
    
    checkpoint_callback = CheckpointCallback(
        save_freq=500_000,
        save_path="./models/checkpoints/",
        name_prefix="mahjong_model"
    )
    
    # Bật verbose=1 để cho phép in text ra màn hình
    stats_callback = SlidingWindowStatsCallback(window_size=10, verbose=1)

    # Gộp thành 1 list duy nhất
    callback_list = CallbackList([checkpoint_callback, stats_callback])

    # ==========================================
    # 4. CHẠY HUẤN LUYỆN (Chỉ 1 lệnh learn duy nhất)
    # ==========================================
    print("=== BẮT ĐẦU HUẤN LUYỆN ===")
    model.learn(
        total_timesteps=2_000_000, 
        progress_bar=False,
        callback=callback_list,  
        tb_log_name="PPO_Adaptive_Penalty_Run_1",
        reset_num_timesteps=False
    )

    # ==========================================
    # 5. LƯU MÔ HÌNH VÀ ĐÓNG MULTIPROCESSING
    # ==========================================
    os.makedirs("models", exist_ok=True)
    model.save("models/maskable_ppo_mahjong_final")
    
    # Quan trọng: Đóng 8 luồng chạy ngầm để giải phóng RAM/CPU
    env.close() 

    # ==========================================
    # 6. CHẠY THỬ VÀ XUẤT REPLAY JSON (MÔI TRƯỜNG ĐƠN)
    # ==========================================
    print("\n=== CHẠY THỬ VÀ XUẤT REPLAY JSON ===")
    
    # Tạo lại 1 bàn cờ duy nhất để AI biểu diễn
    test_env = MahjongEnv()
    test_env = ActionMasker(test_env, mask_fn)
    
    obs, info = test_env.reset()
    done = False
    step_count = 0

    # Khởi tạo cấu trúc dữ liệu Replay
    replay_data = {
        "metadata": {
            "player": "AI_Agent_Maskable",
            "game_type": "1_vs_3_Dummies"
        },
        # Lúc này obs là mảng 1D (68,) nên lệnh lấy 34 phần tử đầu tiên hoàn toàn hợp lệ
        "initial_hand": obs[:34].astype(int).tolist(), 
        "timeline": []
    }

    while not done:
        step_count += 1
        
        # 1. Lấy mask hiện tại và cho AI suy nghĩ
        action_masks = info.get("action_mask")
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        
        # 2. Thực thi hành động vào môi trường biểu diễn
        obs, reward, terminated, truncated, info = test_env.step(action)
        done = terminated or truncated
        
        # 3. Đóng gói dữ liệu của lượt đi này vào lịch sử (timeline)
        turn_log = {
            "turn": step_count,
            "agent_discarded": int(action),
            "dummies_discarded": info.get("dummy_discards", []),
            "agent_drew": info.get("agent_drawn", -1),
            "shanten_after": info["shanten"],
            "wall_left": info["wall_remaining"]
        }
        replay_data["timeline"].append(turn_log)
        
        # In log ra màn hình terminal cho vui mắt
        print(f"Bước {step_count:2d} | Đánh: {action:2d} | Bốc: {info.get('agent_drawn', -1):2d} | Shanten: {info['shanten']} | {info['shanten_detail']['status']}")
        
    # 4. Kết thúc ván đấu và lưu kết quả
    if terminated:
        print("=> AI đã TỚI BÀI!")
        replay_data["result"] = "Winning"
    elif truncated:
        print("=> Hòa (Hết bài).")
        replay_data["result"] = "Draw"

    # 5. Viết toàn bộ dữ liệu ra file
    with open("mahjong_replay.json", "w", encoding="utf-8") as f:
        json.dump(replay_data, f, indent=4, ensure_ascii=False)

    print("=> Toàn bộ diễn biến đã được lưu vào file 'mahjong_replay.json'!")

if __name__ == "__main__":
    main()

In [ ]:
import os
import gymnasium as gym
import numpy as np
import json
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from mahjong_env import MahjongEnv

# 1. Định nghĩa lại hàm lấy Mask (phải giống hệt lúc train)
def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def run_and_export_replay(model_path, output_file="mahjong_replay.json"):
    # 2. Khởi tạo môi trường và bọc Masker
    env = MahjongEnv()
    env = ActionMasker(env, mask_fn)

    # 3. Tải mô hình đã lưu
    if not os.path.exists(model_path):
        print(f"Lỗi: Không tìm thấy file model tại {model_path}")
        return

    print(f"Đang tải model từ: {model_path}...")
    model = MaskablePPO.load(model_path, env=env)

    # 4. Bắt đầu ván đấu
    obs, info = env.reset()
    done = False
    step_count = 0

    # Cấu trúc dữ liệu Replay
    replay_data = {
        "metadata": {
            "model_used": model_path,
            "player": "AI_Agent",
            "date_exported": "2026-04-22"
        },
        "initial_hand": obs[:34].astype(int).tolist(), 
        "timeline": []
    }

    print("\n=== AI BẮT ĐẦU ĐÁNH THỬ 1 VÁN ===")
    
    while not done:
        step_count += 1
        
        # Lấy mask và để AI dự đoán (deterministic=True để AI đánh chuẩn nhất, không bốc đồng)
        action_masks = info.get("action_mask")
        action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
        
        # Thực thi hành động
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # Lưu log lượt đi
        turn_log = {
            "turn": step_count,
            "agent_discarded": int(action),
            "dummies_discarded": info.get("dummy_discards", []),
            "agent_drew": info.get("agent_drawn", -1),
            "shanten_after": info["shanten"],
            "shanten_status": info["shanten_detail"]["status"],
            "reward_this_step": float(reward)
        }
        replay_data["timeline"].append(turn_log)
        
        # In nhanh ra màn hình để theo dõi
        print(f"Lượt {step_count:2d} | AI đánh: {action:2d} | Shanten: {info['shanten']} | Reward: {reward:.2f}")

    # 5. Kết luận ván đấu
    if terminated:
        print("\n🎉 KẾT QUẢ: AI ĐÃ TỚI BÀI (WIN)!")
        replay_data["result"] = "Winning"
    else:
        print("\n🤝 KẾT QUẢ: HÒA (DRAW).")
        replay_data["result"] = "Draw"

    # 6. Xuất ra file JSON
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(replay_data, f, indent=4, ensure_ascii=False)
    
    print(f"\n=> Đã lưu diễn biến ván đấu vào file: {output_file}")

if __name__ == "__main__":
    # Thay đổi đường dẫn tới file model của bạn ở đây
    MODEL_FILE = "models/checkpoints_1/mahjong_model_1000000_steps.zip" 
    run_and_export_replay(MODEL_FILE)

In [ ]:
import os
import gymnasium as gym
import numpy as np
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from mahjong_env import MahjongEnv

# 1. Hàm lấy mask
def mask_fn(env: gym.Env) -> np.ndarray:
    return env.unwrapped._action_mask()

def evaluate_last_5_steps(model_path, num_episodes=100):
    # 2. Khởi tạo môi trường
    env = MahjongEnv()
    env = ActionMasker(env, mask_fn)

    if not os.path.exists(model_path):
        print(f"Lỗi: Không tìm thấy file model tại {model_path}")
        return

    print(f"Đang tải model từ: {model_path}...")
    model = MaskablePPO.load(model_path, env=env)

    # Dictionary để lưu tổng số lần xuất hiện của các mức Shanten trong 5 bước cuối
    # Mình theo dõi từ -1 (Tới bài) cho đến 6 (Rác)
    total_shanten_counts = {i: 0 for i in range(-1, 9)}
    
    print(f"\n🚀 Đang mô phỏng {num_episodes} ván đấu (Vui lòng đợi vài giây)...")

    # 3. Vòng lặp mô phỏng
    for ep in range(num_episodes):
        obs, info = env.reset()
        done = False
        
        # Mảng lưu trữ trạng thái Shanten qua từng bước của ván này
        # Bắt đầu bằng Shanten lúc vừa chia bài xong
        shanten_history = [info['shanten']] 

        while not done:
            action_masks = info.get("action_mask")
            action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
            
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # Ghi nhận Shanten sau mỗi lượt đánh
            shanten_history.append(info['shanten'])

        # 4. Trích xuất 5 bước cuối cùng của ván đấu
        # Nếu ván đấu diễn ra nhanh hơn 5 bước, nó sẽ lấy toàn bộ ván
        last_5_steps = shanten_history[-5:]
        
        # Cộng dồn vào bộ đếm tổng
        for s in last_5_steps:
            if s in total_shanten_counts:
                total_shanten_counts[s] += 1

    # 5. Tính toán trung bình và in kết quả
    print("\n" + "="*50)
    print(f" BÁO CÁO: TẦN SUẤT SHANTEN TRONG 5 BƯỚC CUỐI ({num_episodes} VÁN) ")
    print("="*50)
    print("Mỗi ván có tối đa 5 bước cuối được xét. Dưới đây là\ntrung bình số lượt/ván mà AI đạt mức Shanten tương ứng:\n")

    # In đúng các Shanten từ 1 đến 6 theo yêu cầu
    for shanten_level in range(1, 7):
        # Tính trung bình: Tổng số lượt xuất hiện chia cho số ván
        average_per_episode = total_shanten_counts[shanten_level] / num_episodes
        print(f" 🔹 Shanten {shanten_level}: {average_per_episode:>4.2f} lần / ván")
        
    print("-" * 50)
    # Khuyến mãi thêm thông số của Tenpai và Tới bài để bạn xem tỷ lệ thắng
    avg_tenpai = total_shanten_counts[0] / num_episodes
    avg_win = total_shanten_counts[-1] / num_episodes
    print(f" 🎯 Shanten  0 (Tenpai) : {avg_tenpai:>4.2f} lần / ván")
    print(f" 🏆 Shanten -1 (Tới bài): {avg_win:>4.2f} lần / ván")
    print("="*50)

if __name__ == "__main__":
    # ĐỪNG QUÊN SỬA LẠI ĐƯỜNG DẪN MODEL NẾU CẦN
    MODEL_FILE = "models/checkpoints_1/mahjong_model_1000000_steps.zip" 
    
    # Chạy thử 100 ván
    evaluate_last_5_steps(MODEL_FILE, num_episodes=100)